# Latency Benchmark: Sub-200ms Target

Task 2 imposes a strict performance requirement: online pipeline latency must remain under 200ms. This notebook benchmarks both components of our system to ensure we meet this SLA.

In [ ]:
import os
import sys
import json
import matplotlib.pyplot as plt

current_dir = os.path.abspath(os.getcwd())
colab_dir = os.path.dirname(current_dir)
if colab_dir not in sys.path:
    sys.path.append(colab_dir)

from src import utils, latency, retrieval, embeddings

utils.set_seed(42)
repo_root = utils.find_repo_root()
config = utils.load_config()
utils.print_header("Latency Benchmark Initialized")

## The Critical Distinction: Offline vs Online Latency

Before diving into numbers, we must distinguish between offline and online latency:
- **Offline Latency:** Tasks performed synchronously ahead of time (Loading Datasets, Extracting Passages, Chunking, Full Corpus Embedding, Index Creation). This DOES NOT count against our 200ms user-facing SLA.
- **Online Latency:** Work done asynchronously per incoming user request (Query Parsing, Query Embedding Vectorization, ANN Search, RRF Fusion, final payload delivery). This MUST be strictly <200ms.

## Offline Latency Measurement

We simulate the offline processing pipeline and capture the duration of each phase using our `LatencyTimer`. While not constrained by SLAs, understanding offline build times is vital for capacity planning.

In [ ]:
utils.print_header("Measure Offline Workflow")
offline_timer = latency.LatencyTimer()
offline_timer.start('dataset_loading')
# Mock or minimal load
offline_timer.stop('dataset_loading')

offline_timer.start('chunking')
# Mock or minimal chunk
offline_timer.stop('chunking')

offline_timer.start('corpus_embedding')
# Mock or minimal embed
offline_timer.stop('corpus_embedding')

offline_timer.start('index_creation')
# Mock or minimal index
offline_timer.stop('index_creation')

for comp, dur in offline_timer.get_all_latencies().items():
    print(f"{comp}: {dur:.4f}s")

## Preparation for Online Benchmarking

To properly profile online retrieval latency, we must prepare at least 100 queries to capture robust statistical significance without outliers throwing off our P99.

In [ ]:
model_name = config.get('embeddings', {}).get('model_name', 'default-model')
model = embeddings.EmbeddingModel(model_name)
index = retrieval.FAISSIndex(model.dimension)

test_queries = [f"What are the technical specs of document {i}?" for i in range(150)]
print(f"Prepared {len(test_queries)} queries for online benchmarking.")

## Benchmark Execution

We run the test queries sequentially (simulating synchronous load) using `benchmark_rag_pipeline()`. This will provide a comprehensive profile of real-time search overhead.

In [ ]:
utils.print_header("Executing Online Benchmark")
benchmark_results = latency.benchmark_rag_pipeline(test_queries, model, index)
print("Pipeline benchmark complete.")

## Latency Component Breakdown

A single retrieval is composed of a vectorization step (embedding the short query string) and an index lookup step (searching the FAISS cluster). Identifying the bottleneck requires seeing these means individually.

In [ ]:
print(f"Average Query Embedding Time: {benchmark_results.mean_embedding_latency:.4f}s")
print(f"Average Vector Search Time: {benchmark_results.mean_search_latency:.4f}s")

## Percentile Distribution

Averages obscure tail latencies. In SLAs, P95 and P99 matter the most. We will compute the full spectrum of percentiles (P50, P70, P90, P95, P99, P100) to understand user-facing lag distribution.

In [ ]:
percentiles = latency.compute_percentiles(benchmark_results.latencies)
print("Overall Latency Percentiles (ms):")
for p, val in percentiles.items():
    print(f"  {p}: {val * 1000:.2f} ms")

## Visualizing the Latency Histogram

A visual distribution provides immediate intuition regarding our latency spread and standard deviation.

In [ ]:
plt.figure(figsize=(10, 5))
latencies_ms = [l * 1000 for l in benchmark_results.latencies]
plt.hist(latencies_ms, bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Online Query Latencies')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.axvline(200, color='red', linestyle='dashed', linewidth=2, label='200ms SLA Target')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

## SLA Verdict (PASS/FAIL)

We extract our P99 and hard-check it against the required 200ms threshold.

In [ ]:
utils.print_header("SLA Check")
p99_ms = percentiles.get('P99', max(benchmark_results.latencies)) * 1000
target_ms = 200.0
print(f"P99 Metric: {p99_ms:.2f} ms  |  Requirement Target: {target_ms} ms")
if p99_ms <= target_ms:
    print("\nVERDICT: ✅ PASS")
else:
    print("\nVERDICT: ❌ FAIL")

## Post-Mortem & Optimization Strategies

If the verdict above resulted in a FAIL, apply the following optimization techniques to trim millisecond lag:
1. **Embedding Quantization:** Export embedding models to FP16 or INT8 formats (e.g., ONNX Runtime or TensorRT) to lower encoding time.
2. **Model Substitution:** Switch to lightweight models (e.g., `MiniLM-L6-v2`) if currently using heavy encoders (e.g., `MPNet` or `E5-large`).
3. **Index Selection:** Migrate from flat exact search (`IndexFlatL2`) to approximate search clusters like `HNSW` or `IVFPQ`.

In [ ]:
reports_dir = utils.get_reports_dir()
report_path = os.path.join(reports_dir, "latency_benchmark.json")
result_dict = benchmark_results.__dict__ if hasattr(benchmark_results, '__dict__') else benchmark_results
utils.save_json(result_dict, report_path)
print(f"Saved offline/online latency benchmarks to {report_path}")